In [1]:
import anndata as ad
import json
import pandas as pd


In [2]:

adata = ad.read_h5ad("/scratch/2370352/my-research/data/censusxgene/breast_norm_selected.h5ad")

In [3]:
print(adata)
print("obs columns:", adata.obs.columns)
print("var shape:", adata.var.shape)
print("X type:", type(adata.X))

AnnData object with n_obs × n_vars = 28000 × 61497
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
obs columns: Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id

In [4]:
census_genes = adata.var['feature_name'].tolist()

print(f"Liczba genów w adata: {len(census_genes)}")
print(census_genes[:10])  # pierwsze 10 genów

Liczba genów w adata: 61497
['LINC01409', 'NOC2L', 'PERM1', 'ENSG00000272512', 'HES4', 'ISG15', 'AGRN', 'RNF223', 'C1orf159', 'TNFRSF18']


In [5]:
vocab_path = '/scratch/2370352/my-research/papers/scgpt/save/whole_human/vocab.json'

with open(vocab_path, "r") as f:
    vocab = json.load(f)

model_genes = list(vocab.keys())

print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(model_genes[:10])  # przykładowe pierwsze 10 genów

Liczba genów w scGPT vocab: 60697
['RP5-973N23.5', 'RP11-182N22.10', 'CTB-53D8.3', 'RP11-348N17.2', 'RP11-205M20.8', 'RP11-326C3.17', 'RP11-439H13.3', 'RP11-413H22.3', 'GET1-SH3BGR', 'CH17-476P10.1']


In [6]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = adata.var['feature_name'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 61497
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 38598
Liczba genów w adata nie w vocab: 21430
Liczba genów w vocab nie w adata: 22099
Przykłady genów wspólnych: ['RNU2-50P', 'RHOH', 'HPCAL4', 'SNORD11B', 'YIPF3', 'GTDC1', 'RN7SKP26', 'RN7SL665P', 'LRRC74A', 'TPM1']
Przykłady genów w adata ale nie w vocab: ['ENSG00000278884', 'LILRA2P1', 'ENSG00000225170', 'ENSG00000278239', 'ENSG00000253706', 'ENSG00000278961', 'ENSG00000239793', 'ENSG00000272597', 'ENSG00000261093', 'ENSG00000223908']
Przykłady genów w vocab ale nie w adata: ['RP11-1072C15.6', 'RP11-1437A8.2', 'AC098784.1', 'LL0XNC01-36H8.1', 'RP11-1D19.1', 'Y_RNA_ENSG00000252115', 'AP001437.2', 'RP5-1132H15.2', 'RP4-714D9.2', 'RP11-486O13.3']


In [7]:
gene_info = pd.read_csv("/scratch/2370352/my-research/data/gene_info_table.csv")  # lub pełna ścieżka

# Stwórz słownik: ensembl_id -> gene_name
ensg_to_symbol = dict(zip(gene_info['ensembl_id'], gene_info['gene_name']))

print(list(ensg_to_symbol.items())[:10])

[('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1'), ('ENSG00000000457', 'SCYL3'), ('ENSG00000000460', 'C1orf112'), ('ENSG00000000938', 'FGR'), ('ENSG00000000971', 'CFH'), ('ENSG00000001036', 'FUCA2'), ('ENSG00000001084', 'GCLC'), ('ENSG00000001167', 'NFYA')]


In [8]:
# jeśli w adata masz geny zapisane jako ENSG, mapujemy je
mapped_genes = []

for g in adata.var['feature_name']:
    if g in ensg_to_symbol:
        mapped_genes.append(ensg_to_symbol[g])
    else:
        mapped_genes.append(g)  # zachowaj tak jak jest, np. już symboliczny gen

# podmieniamy w adata.var
adata.var['feature_name_mapped'] = mapped_genes

In [9]:
# lista genów po mapowaniu
adata_genes_mapped = adata.var['feature_name_mapped'].tolist()
adata_set = set(adata_genes_mapped)
model_set = set(model_genes)  # z vocab scGPT

common_genes = adata_set & model_set
missing_in_model = adata_set - model_set
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata po mapowaniu: {len(adata_genes_mapped)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata ale nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab ale nie w adata: {len(missing_in_adata)}")

print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata po mapowaniu: 61497
Liczba genów wspólnych: 47856
Liczba genów w adata ale nie w vocab: 11611
Liczba genów w vocab ale nie w adata: 12841
Przykłady genów wspólnych: ['RP11-1437A8.2', 'RNU2-50P', 'AC098784.1', 'RHOH', 'HPCAL4', 'YIPF3', 'SNORD11B', 'GTDC1', 'RN7SKP26', 'RN7SL665P']
Przykłady genów w adata ale nie w vocab: [nan, 'LILRA2P1', 'DGAT2-DT', 'RP11-255H23.2', 'ENSG00000287546', 'BX546450.2', 'ENSG00000290068', 'CU639417.4', 'AC010998.2', 'ENSG00000286225']
Przykłady genów w vocab ale nie w adata: ['RP11-1072C15.6', 'RP11-495L18.2', 'AC012442.6', 'RP11-723O4.10', 'RP11-791G22.3', 'LL0XNC01-36H8.1', 'RP11-529G21.2', 'RP11-164O23.8', 'RP11-496I2.8', 'RP11-1D19.1']


In [10]:
# Słownik gene_name -> gene_type
gene_type_dict = dict(zip(gene_info['gene_name'], gene_info['gene_type']))

# Lista genów po mapowaniu (feature_name_mapped)
mapped_genes = adata.var['feature_name_mapped'].tolist()

# Geny wspólne z scGPT
common_genes_set = set(mapped_genes) & set(model_genes)
# Geny w adata ale nie w vocab
not_in_vocab_set = set(mapped_genes) - set(model_genes)

# Funkcja licząca protein-coding vs reszta
def count_gene_types(gene_list):
    protein_coding = 0
    non_coding = 0
    unknown = 0
    for g in gene_list:
        t = gene_type_dict.get(g, None)
        if t == "protein_coding":
            protein_coding += 1
        elif t is None:
            unknown += 1
        else:
            non_coding += 1
    return protein_coding, non_coding, unknown

# Liczby dla genów wspólnych
pc_common, nc_common, unk_common = count_gene_types(common_genes_set)
# Liczby dla genów niezmapowanych do vocab
pc_not, nc_not, unk_not = count_gene_types(not_in_vocab_set)

print("Wspólne geny z vocab scGPT:")
print(f"Protein-coding: {pc_common}, Non-coding: {nc_common}, Unknown: {unk_common}")

print("\nGeny w adata ale nie w vocab scGPT:")
print(f"Protein-coding: {pc_not}, Non-coding: {nc_not}, Unknown: {unk_not}")

Wspólne geny z vocab scGPT:
Protein-coding: 18229, Non-coding: 24008, Unknown: 5619

Geny w adata ale nie w vocab scGPT:
Protein-coding: 329, Non-coding: 5009, Unknown: 6273
